In [1]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Users\HP-ZBOOK i7\graphrag_test")

pilot1_relationships = pd.read_csv(
    project_root / "pilot_01" / "inspection" / "04_relationships_all.csv"
)

pilot2_relationships = pd.read_csv(
    project_root / "pilot_02" / "inspection" / "04_relationships_all.csv"
)

print("Pilot 1 relationships:", len(pilot1_relationships))
print("Pilot 2 relationships:", len(pilot2_relationships))

absolute_change = len(pilot2_relationships) - len(pilot1_relationships)
percentage_change = (
    absolute_change / len(pilot1_relationships) * 100
)

print("Absolute change:", absolute_change)
print("Percentage change:", round(percentage_change, 1), "%")

print("\nPilot 2 columns:")
print(pilot2_relationships.columns.tolist())

Pilot 1 relationships: 414
Pilot 2 relationships: 643
Absolute change: 229
Percentage change: 55.3 %

Pilot 2 columns:
['id', 'human_readable_id', 'source', 'target', 'description', 'weight', 'combined_degree', 'text_unit_ids']


In [2]:
pd.set_option("display.max_colwidth", 140)

pilot2_relationships[
    [
        "source",
        "target",
        "description",
        "weight",
        "combined_degree",
        "text_unit_ids",
    ]
].sort_values(
    by=["combined_degree", "weight"],
    ascending=False,
).head(30)

,source,target,description,weight,combined_degree,text_unit_ids
194,PHOTOVOLTAIK,ÖSTERREICH,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLICIT_FACT] Photovoltaics is named as a key technology in Austria's future energy supply mix.,10.0,137,['f9e54ba797ff2b80a5f0e5f24e3901236ac4130e4e84d01615e67bc810b290cd5ac7c08f71a782797073050d512eda2695c3659569deba0d4bd5c3ab6195784a']
2,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,PHOTOVOLTAIK,The Austrian Photovoltaic Strategy (ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE) is a guiding framework dedicated to the expansion and strate...,30.0,130,['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475'\n '5...
616,PHOTOVOLTAIK,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPLICIT_FACT] The Austrian Photovoltaic Strategy concerns the deployment of photovoltaics in ...,9.0,130,['6927535991c09031efd0df39ec21591dd3bb460da14c01bc36d343f0078b726ed44d06a0c6bb518236ea61ed8c8b15365630cf185b8012e963898f2e117f7d8a']
610,PHOTOVOLTAIK,PV-ANLAGE,PV-Anlage (PV systems) are a primary and central form of Photovoltaik (photovoltaics) installations.,10.0,111,['6927535991c09031efd0df39ec21591dd3bb460da14c01bc36d343f0078b726ed44d06a0c6bb518236ea61ed8c8b15365630cf185b8012e963898f2e117f7d8a'\n '6...
498,ÖSTERREICH,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,The ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE (Austrian Photovoltaic Strategy) is a national policy instrument in Austria that is associate...,20.0,97,['9ce049a98570c6bf53cf9e547168db075af2795fa519422bd27a2c9aa5b490c4b26aef25883802c4d8eb6e4dcb09c866e7d31677352f7beb6c4f9fcbbcc75e9b'\n '0...
198,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,ÖSTERREICH,The Austrian Photovoltaic Strategy (ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE) is applicable to Austria's national PV development and perta...,17.0,97,['f9e54ba797ff2b80a5f0e5f24e3901236ac4130e4e84d01615e67bc810b290cd5ac7c08f71a782797073050d512eda2695c3659569deba0d4bd5c3ab6195784a'\n '6...
338,PHOTOVOLTAIK,PHOTOVOLTAIK-ERZEUGUNGSSPITZEN,"[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLICIT_FACT] Expansion of photovoltaics leads to production peaks, which are already challeng...",9.0,93,['6a1a5132f360531b1174e0eac2b2c5b6ece3b0771a8037231306baf0af8e8ca23de435f581fc62d6794cab81a5dadf3235b66eda56c331a0043621c688fbe302']
375,PHOTOVOLTAIK,PHOTOVOLTAIKANLAGEN,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPLICIT_FACT] Photovoltaics is the general technology class under which photovoltaic installa...,9.0,93,['2d9fdebd364caf6ea226dc0ee6624e0c2d54fc65848a9cd18f9b868675333d5be302c20a6cbb1b4f9503e536f5051a60a8783cd3657ab2cf1f9daf65be366c76']
300,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP),PHOTOVOLTAIK,"[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FACT] The NIP identifies and supports area potentials on buildings, infrastructures, and ope...",8.0,93,['41bd25afdc1cce74b36dd6db5e44e91354172fda16475ab6a3f6cf99bddc720aa91cb56b5ea1b1c4774c8c97017d6bb03c52ba2676d7cfaab1c73fed8ab82912']
17,ENERGIEWENDE,PHOTOVOLTAIK,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FACT] The energy transition in Austria requires the expansion of photovoltaics as a key rene...,10.0,91,['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475']


In [3]:
import re

pilot2_entities = pd.read_csv(
    project_root / "pilot_02" / "inspection" / "01_entities_all.csv"
)

# Map every entity title to its Pilot 2 type
entity_type_map = (
    pilot2_entities
    .drop_duplicates(subset="title")
    .set_index("title")["type"]
    .to_dict()
)

relationship_analysis = pilot2_relationships.copy()

relationship_analysis["source_type"] = (
    relationship_analysis["source"]
    .map(entity_type_map)
    .fillna("UNKNOWN")
)

relationship_analysis["target_type"] = (
    relationship_analysis["target"]
    .map(entity_type_map)
    .fillna("UNKNOWN")
)

# Extract the controlled relationship label from the description
relationship_analysis["relation_type"] = (
    relationship_analysis["description"]
    .str.extract(r"\[RELATION_TYPE=([^\]]+)\]", expand=False)
    .fillna("UNSPECIFIED")
)

# Extract whether the statement is a fact, recommendation, target, scenario, etc.
relationship_analysis["modality"] = (
    relationship_analysis["description"]
    .str.extract(r"\[MODALITY=([^\]]+)\]", expand=False)
    .fillna("UNSPECIFIED")
)

print("Relationship types:")
print(relationship_analysis["relation_type"].value_counts())

print("\nModalities:")
print(relationship_analysis["modality"].value_counts())

print("\nEndpoint entity types:")
print(
    pd.concat(
        [
            relationship_analysis["source_type"],
            relationship_analysis["target_type"],
        ]
    ).value_counts()
)

Relationship types:
relation_type
SUPPORTS           164
ASSOCIATED_WITH    107
CONTRIBUTES_TO      70
PROPOSES            44
PART_OF             39
MEASURED_BY         37
CONSTRAINS          33
RESPONSIBLE_FOR     25
SETS_TARGET         18
HAS_TARGET          18
REGULATES           15
IMPLEMENTS          11
SUPPORTED_BY        11
REQUIRES            10
UNSPECIFIED          9
LOCATED_IN           9
ALIAS_OF             7
FUNDS                6
APPLIES_TO           4
IMPLEMENTED_BY       2
BENEFITS_FROM        1
IMPLEMENTED          1
CAN_USE              1
AMENDS               1
Name: count, dtype: int64

Modalities:
modality
EXPLICIT_FACT           550
PLANNED_ACTION           26
RECOMMENDATION           19
SCENARIO                 18
PROPOSAL                 13
UNSPECIFIED               9
LEGAL_REQUIREMENT         5
REASONABLE_INFERENCE      3
Name: count, dtype: int64

Endpoint entity types:
TECHNOLOGY         283
POLICY             265
TARGET             126
STAKEHOLDER        125


In [4]:
# 1. Highly connected relationships
top_relationships = (
    relationship_analysis
    .sort_values(
        by=["combined_degree", "weight"],
        ascending=False,
    )
    .head(25)
)

# 2. Relationships involving targets or market metrics
quantitative_relationships = relationship_analysis[
    relationship_analysis["source_type"].isin(
        ["TARGET", "MARKET_METRIC"]
    )
    |
    relationship_analysis["target_type"].isin(
        ["TARGET", "MARKET_METRIC"]
    )
].sort_values(
    by=["combined_degree", "weight"],
    ascending=False,
).head(20)

# 3. Stratified sample from every relationship type
stratified_relationships = pd.concat(
    [
        group.sample(
            n=min(2, len(group)),
            random_state=42,
        )
        for _, group in relationship_analysis.groupby("relation_type")
    ],
    ignore_index=True,
)

# Combine and remove repeated relationships
relationship_audit_sample = (
    pd.concat(
        [
            top_relationships,
            quantitative_relationships,
            stratified_relationships,
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset="id")
    .reset_index(drop=True)
)

# Fill to exactly 60 if overlapping selections reduced the sample
if len(relationship_audit_sample) < 60:
    remaining_relationships = relationship_analysis[
        ~relationship_analysis["id"].isin(
            relationship_audit_sample["id"]
        )
    ]

    additional_relationships = remaining_relationships.sample(
        n=60 - len(relationship_audit_sample),
        random_state=42,
    )

    relationship_audit_sample = (
        pd.concat(
            [
                relationship_audit_sample,
                additional_relationships,
            ],
            ignore_index=True,
        )
        .reset_index(drop=True)
    )

# Limit to 60 if the stratified selection produced more
relationship_audit_sample = (
    relationship_audit_sample
    .head(60)
    .reset_index(drop=True)
)

print("Relationship audit sample size:",
      len(relationship_audit_sample))

print("\nRelationship types in sample:")
print(
    relationship_audit_sample["relation_type"]
    .value_counts()
)

print("\nModalities in sample:")
print(
    relationship_audit_sample["modality"]
    .value_counts()
)

print("\nRelationships involving TARGET or MARKET_METRIC:")
print(
    (
        relationship_audit_sample["source_type"].isin(
            ["TARGET", "MARKET_METRIC"]
        )
        |
        relationship_audit_sample["target_type"].isin(
            ["TARGET", "MARKET_METRIC"]
        )
    ).sum()
)

Relationship audit sample size: 60

Relationship types in sample:
relation_type
CONTRIBUTES_TO     17
MEASURED_BY         8
SUPPORTS            7
ASSOCIATED_WITH     5
UNSPECIFIED         4
CONSTRAINS          3
ALIAS_OF            2
FUNDS               2
APPLIES_TO          2
IMPLEMENTED_BY      2
HAS_TARGET          2
AMENDS              1
REGULATES           1
CAN_USE             1
BENEFITS_FROM       1
IMPLEMENTED         1
IMPLEMENTS          1
Name: count, dtype: int64

Modalities in sample:
modality
EXPLICIT_FACT     52
UNSPECIFIED        4
SCENARIO           3
RECOMMENDATION     1
Name: count, dtype: int64

Relationships involving TARGET or MARKET_METRIC:
26


In [5]:
relationship_review = relationship_audit_sample[
    [
        "id",
        "source",
        "source_type",
        "target",
        "target_type",
        "relation_type",
        "modality",
        "description",
        "weight",
        "combined_degree",
        "text_unit_ids",
    ]
].copy()

relationship_review["source_supported"] = ""
relationship_review["primary_classification"] = ""
relationship_review["relation_type_correct"] = ""
relationship_review["modality_correct"] = ""
relationship_review["direction_correct"] = ""
relationship_review["endpoints_correct"] = ""
relationship_review["description_problem"] = ""
relationship_review["likely_pipeline_stage"] = ""
relationship_review["audit_notes"] = ""

relationship_review.head(10)

,id,source,source_type,target,target_type,relation_type,modality,description,weight,combined_degree,text_unit_ids,source_supported,primary_classification,relation_type_correct,modality_correct,direction_correct,endpoints_correct,description_problem,likely_pipeline_stage,audit_notes
0,38aaae30-345b-449c-a1d7-f36502b38194,PHOTOVOLTAIK,TECHNOLOGY,ÖSTERREICH,GEOGRAPHIC_AREA,CONTRIBUTES_TO,EXPLICIT_FACT,[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLICIT_FACT] Photovoltaics is named as a key technology in Austria's future energy supply mix.,10.0,137,['f9e54ba797ff2b80a5f0e5f24e3901236ac4130e4e84d01615e67bc810b290cd5ac7c08f71a782797073050d512eda2695c3659569deba0d4bd5c3ab6195784a'],,,,,,,,,
1,83f96aa2-16da-43bb-be18-8e0fc0c0ba48,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,PHOTOVOLTAIK,TECHNOLOGY,UNSPECIFIED,UNSPECIFIED,The Austrian Photovoltaic Strategy (ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE) is a guiding framework dedicated to the expansion and strate...,30.0,130,['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475'\n '5...,,,,,,,,,
2,ba0af85a-6483-445b-9414-a781ddea46e5,PHOTOVOLTAIK,TECHNOLOGY,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,ASSOCIATED_WITH,EXPLICIT_FACT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPLICIT_FACT] The Austrian Photovoltaic Strategy concerns the deployment of photovoltaics in ...,9.0,130,['6927535991c09031efd0df39ec21591dd3bb460da14c01bc36d343f0078b726ed44d06a0c6bb518236ea61ed8c8b15365630cf185b8012e963898f2e117f7d8a'],,,,,,,,,
3,c9e4c4d1-659a-43c6-83bd-6804552a6017,PHOTOVOLTAIK,TECHNOLOGY,PV-ANLAGE,TECHNOLOGY,UNSPECIFIED,UNSPECIFIED,PV-Anlage (PV systems) are a primary and central form of Photovoltaik (photovoltaics) installations.,10.0,111,['6927535991c09031efd0df39ec21591dd3bb460da14c01bc36d343f0078b726ed44d06a0c6bb518236ea61ed8c8b15365630cf185b8012e963898f2e117f7d8a'\n '6...,,,,,,,,,
4,193accbf-de49-42e5-8e40-f1aa91f9890e,ÖSTERREICH,GEOGRAPHIC_AREA,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,UNSPECIFIED,UNSPECIFIED,The ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE (Austrian Photovoltaic Strategy) is a national policy instrument in Austria that is associate...,20.0,97,['9ce049a98570c6bf53cf9e547168db075af2795fa519422bd27a2c9aa5b490c4b26aef25883802c4d8eb6e4dcb09c866e7d31677352f7beb6c4f9fcbbcc75e9b'\n '0...,,,,,,,,,
5,0090f21b-a13b-4695-bc3d-1e48b29cd8aa,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,ÖSTERREICH,GEOGRAPHIC_AREA,UNSPECIFIED,UNSPECIFIED,The Austrian Photovoltaic Strategy (ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE) is applicable to Austria's national PV development and perta...,17.0,97,['f9e54ba797ff2b80a5f0e5f24e3901236ac4130e4e84d01615e67bc810b290cd5ac7c08f71a782797073050d512eda2695c3659569deba0d4bd5c3ab6195784a'\n '6...,,,,,,,,,
6,604c93cd-7c69-467b-828c-13e257b5d3d7,PHOTOVOLTAIK,TECHNOLOGY,PHOTOVOLTAIK-ERZEUGUNGSSPITZEN,CONSTRAINT,CONTRIBUTES_TO,EXPLICIT_FACT,"[RELATION_TYPE=CONTRIBUTES_TO] [MODALITY=EXPLICIT_FACT] Expansion of photovoltaics leads to production peaks, which are already challeng...",9.0,93,['6a1a5132f360531b1174e0eac2b2c5b6ece3b0771a8037231306baf0af8e8ca23de435f581fc62d6794cab81a5dadf3235b66eda56c331a0043621c688fbe302'],,,,,,,,,
7,ee657e2c-7cc4-4b96-9c32-dff740b2f9a3,PHOTOVOLTAIK,TECHNOLOGY,PHOTOVOLTAIKANLAGEN,TECHNOLOGY,ASSOCIATED_WITH,EXPLICIT_FACT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPLICIT_FACT] Photovoltaics is the general technology class under which photovoltaic installa...,9.0,93,['2d9fdebd364caf6ea226dc0ee6624e0c2d54fc65848a9cd18f9b868675333d5be302c20a6cbb1b4f9503e536f5051a60a8783cd3657ab2cf1f9daf65be366c76'],,,,,,,,,
8,2867e2c3-10e4-4bb9-b0eb-e10c59152541,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP),POLICY,PHOTOVOLTAIK,TECHNOLOGY,SUPPORTS,EXPLICIT_FACT,"[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FACT] The NIP identifies and supports area potentials on buildings, infrastructures, and ope...",8.0,93,['41bd25afdc1cce74b36dd6db5e44e91354172fda16475ab6a3f6cf99bddc720aa91cb56b5ea1

In [7]:
sample_path = (
    project_root
    / "pilot_02"
    / "inspection"
    / "05_relationship_review_sample.csv"
)

relationship_review.to_csv(
    sample_path,
    index=False,
    encoding="utf-8-sig",
)

print("Relationship-review sample saved to:")
print(sample_path)

Relationship-review sample saved to:
C:\Users\HP-ZBOOK i7\graphrag_test\pilot_02\inspection\05_relationship_review_sample.csv


In [8]:
def audit_relationship(
    row_number,
    source_supported,
    classification,
    relation_type_correct,
    modality_correct,
    direction_correct,
    endpoints_correct,
    description_problem="",
    pipeline_stage="NONE",
    notes="",
):
    relationship_review.loc[
        row_number, "source_supported"
    ] = source_supported

    relationship_review.loc[
        row_number, "primary_classification"
    ] = classification

    relationship_review.loc[
        row_number, "relation_type_correct"
    ] = relation_type_correct

    relationship_review.loc[
        row_number, "modality_correct"
    ] = modality_correct

    relationship_review.loc[
        row_number, "direction_correct"
    ] = direction_correct

    relationship_review.loc[
        row_number, "endpoints_correct"
    ] = endpoints_correct

    relationship_review.loc[
        row_number, "description_problem"
    ] = description_problem

    relationship_review.loc[
        row_number, "likely_pipeline_stage"
    ] = pipeline_stage

    relationship_review.loc[
        row_number, "audit_notes"
    ] = notes

In [9]:
audit_relationship(
    0,
    "PARTIAL",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "NO",
    "PARTIAL",
    "PARTIAL",
    "NO",
    description_problem="The source states that PV contributes to Austria's future energy supply, not that PV contributes to Austria as a geographic entity.",
    pipeline_stage="RELATIONSHIP_ENDPOINT_SELECTION",
    notes="A better target would be FUTURE_ENERGY_SUPPLY or ENERGY_MIX rather than ÖSTERREICH."
)

audit_relationship(
    1,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "NO",
    "NO",
    "YES",
    "YES",
    description_problem="The relationship is source-supported, but GraphRAG left both relation type and modality unspecified.",
    pipeline_stage="RELATIONSHIP_LABEL_EXTRACTION",
    notes="The strategy explicitly provides objectives and measures for PV expansion. A suitable type would be SUPPORTS or SETS_TARGET_FOR."
)

audit_relationship(
    2,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The document explicitly concerns photovoltaic development. ASSOCIATED_WITH is broad but acceptable."
)

audit_relationship(
    3,
    "YES",
    "REASONABLE_INFERENCE_NOT_EXPLICIT",
    "NO",
    "NO",
    "YES",
    "YES",
    description_problem="The taxonomic statement that PV-ANLAGE is a specific form of PHOTOVOLTAIK is reasonable, but it is not expressed as a formal relationship in the source.",
    pipeline_stage="RELATIONSHIP_INFERENCE",
    notes="A more precise controlled relationship would be INSTANCE_OR_FORM_OF rather than UNSPECIFIED."
)

audit_relationship(
    4,
    "YES",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "NO",
    "NO",
    "NO",
    "YES",
    description_problem="The strategy applies to Austria; Austria does not apply to or govern the strategy in the direction shown.",
    pipeline_stage="RELATIONSHIP_DIRECTION_AND_LABEL",
    notes="A better representation is ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE APPLIES_TO ÖSTERREICH."
)

audit_relationship(
    5,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "NO",
    "NO",
    "YES",
    "YES",
    description_problem="The endpoints and direction are appropriate, but the relation type and modality are unspecified.",
    pipeline_stage="RELATIONSHIP_LABEL_EXTRACTION",
    notes="A suitable controlled relationship is APPLIES_TO with EXPLICIT_FACT modality."
)

audit_relationship(
    6,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    description_problem="CONTRIBUTES_TO is acceptable, although CAUSES_OR_INCREASES would express the relationship more precisely.",
    pipeline_stage="NONE",
    notes="The source explicitly discusses PV generation peaks and difficulties absorbing them in the public grid."
)

audit_relationship(
    7,
    "YES",
    "REASONABLE_INFERENCE_NOT_EXPLICIT",
    "YES",
    "NO",
    "YES",
    "YES",
    description_problem="The relationship is a reasonable taxonomic interpretation rather than an explicit source claim.",
    pipeline_stage="RELATIONSHIP_INFERENCE",
    notes="Photovoltaic installations are a deployment form of photovoltaics, but EXPLICIT_FACT overstates the source status."
)

audit_relationship(
    8,
    "PARTIAL",
    "OVERGENERALIZED",
    "PARTIAL",
    "NO",
    "YES",
    "YES",
    description_problem="The NIP supports PV planning and identifies PV potential, but the description broadly attributes building, infrastructure and open-space area potentials to the NIP.",
    pipeline_stage="RELATIONSHIP_DESCRIPTION_SUMMARIZATION",
    notes="The central NIP–PV connection is valid, but its stated scope is broader than the directly supporting text."
)

audit_relationship(
    9,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "NO",
    "YES",
    "YES",
    "YES",
    description_problem="The source supports the statement, but REQUIRES is more accurate than SUPPORTS.",
    pipeline_stage="RELATIONSHIP_TYPE_SELECTION",
    notes="The energy transition is described as requiring renewable-energy and photovoltaic expansion."
)

In [10]:
reviewed_relationships = relationship_review[
    relationship_review["primary_classification"] != ""
]

print("Reviewed relationships:", len(reviewed_relationships))

reviewed_relationships[
    [
        "source",
        "relation_type",
        "target",
        "modality",
        "source_supported",
        "primary_classification",
        "relation_type_correct",
        "modality_correct",
        "direction_correct",
        "endpoints_correct",
        "audit_notes",
    ]
]

Reviewed relationships: 10


,source,relation_type,target,modality,source_supported,primary_classification,relation_type_correct,modality_correct,direction_correct,endpoints_correct,audit_notes
0,PHOTOVOLTAIK,CONTRIBUTES_TO,ÖSTERREICH,EXPLICIT_FACT,PARTIAL,WRONG_DIRECTION_OR_ENDPOINTS,NO,PARTIAL,PARTIAL,NO,A better target would be FUTURE_ENERGY_SUPPLY or ENERGY_MIX rather than ÖSTERREICH.
1,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,UNSPECIFIED,PHOTOVOLTAIK,UNSPECIFIED,YES,DIRECTLY_SUPPORTED_BY_SOURCE,NO,NO,YES,YES,The strategy explicitly provides objectives and measures for PV expansion. A suitable type would be SUPPORTS or SETS_TARGET_FOR.
2,PHOTOVOLTAIK,ASSOCIATED_WITH,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,EXPLICIT_FACT,YES,DIRECTLY_SUPPORTED_BY_SOURCE,YES,YES,YES,YES,The document explicitly concerns photovoltaic development. ASSOCIATED_WITH is broad but acceptable.
3,PHOTOVOLTAIK,UNSPECIFIED,PV-ANLAGE,UNSPECIFIED,YES,REASONABLE_INFERENCE_NOT_EXPLICIT,NO,NO,YES,YES,A more precise controlled relationship would be INSTANCE_OR_FORM_OF rather than UNSPECIFIED.
4,ÖSTERREICH,UNSPECIFIED,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,UNSPECIFIED,YES,WRONG_DIRECTION_OR_ENDPOINTS,NO,NO,NO,YES,A better representation is ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE APPLIES_TO ÖSTERREICH.
5,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,UNSPECIFIED,ÖSTERREICH,UNSPECIFIED,YES,DIRECTLY_SUPPORTED_BY_SOURCE,NO,NO,YES,YES,A suitable controlled relationship is APPLIES_TO with EXPLICIT_FACT modality.
6,PHOTOVOLTAIK,CONTRIBUTES_TO,PHOTOVOLTAIK-ERZEUGUNGSSPITZEN,EXPLICIT_FACT,YES,DIRECTLY_SUPPORTED_BY_SOURCE,YES,YES,YES,YES,The source explicitly discusses PV generation peaks and difficulties absorbing them in the public grid.
7,PHOTOVOLTAIK,ASSOCIATED_WITH,PHOTOVOLTAIKANLAGEN,EXPLICIT_FACT,YES,REASONABLE_INFERENCE_NOT_EXPLICIT,YES,NO,YES,YES,"Photovoltaic installations are a deployment form of photovoltaics, but EXPLICIT_FACT overstates the source status."
8,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP),SUPPORTS,PHOTOVOLTAIK,EXPLICIT_FACT,PARTIAL,OVERGENERALIZED,PARTIAL,NO,YES,YES,"The central NIP–PV connection is valid, but its stated scope is broader than the directly supporting text."
9,ENERGIEWENDE,SUPPORTS,PHOTOVOLTAIK,EXPLICIT_FACT,YES,DIRECTLY_SUPPORTED_BY_SOURCE,NO,YES,YES,YES,The energy transition is described as requiring renewable-energy and photovoltaic expansion.


In [11]:
audit_relationship(
    10,
    "YES",
    "REASONABLE_INFERENCE_NOT_EXPLICIT",
    "PARTIAL",
    "NO",
    "YES",
    "YES",
    description_problem="The source states that the estimated PV potential is 41 TWh by 2040, but does not explicitly phrase PV as contributing to its own potential.",
    pipeline_stage="RELATIONSHIP_INFERENCE",
    notes="A more precise representation would connect the NIP or Austria to the 41 TWh target using HAS_TARGET or IDENTIFIES_POTENTIAL."
)

audit_relationship(
    11,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly describes energy communities as a basis for communal photovoltaic expansion."
)

audit_relationship(
    12,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly identifies photovoltaics as an important pillar for achieving the renewable-electricity objective."
)

audit_relationship(
    13,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The 21 TWh requirement for 2030 is presented within the Transition 2040 scenario, so SCENARIO is appropriate."
)

audit_relationship(
    14,
    "PARTIAL",
    "REASONABLE_INFERENCE_NOT_EXPLICIT",
    "PARTIAL",
    "NO",
    "YES",
    "YES",
    description_problem="The source describes PV as highly decentralized, but the claim that it supports household and business energy use is a synthesized interpretation.",
    pipeline_stage="RELATIONSHIP_INFERENCE",
    notes="The decentralization connection is reasonable, but EXPLICIT_FACT is too strong."
)

audit_relationship(
    15,
    "YES",
    "OVERGENERALIZED",
    "YES",
    "YES",
    "YES",
    "YES",
    description_problem="The description calls 11 TWh generation capacity. TWh measures energy generation, not installed capacity.",
    pipeline_stage="RELATIONSHIP_DESCRIPTION_SUMMARIZATION",
    notes="The numerical target is source-supported, but the generated description confuses energy and capacity."
)

audit_relationship(
    16,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly presents PV expansion as a major component of achieving Austria's 2040 climate-neutrality objective."
)

audit_relationship(
    17,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "NO",
    "YES",
    "YES",
    description_problem="The acceleration areas originate from an EU legal-policy provision rather than merely a general recommendation.",
    pipeline_stage="MODALITY_CLASSIFICATION",
    notes="SUPPORTS is suitable, but LEGAL_REQUIREMENT or PLANNED_ACTION would describe the modality more accurately."
)

audit_relationship(
    18,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly reports 6.3 TWh of Austrian PV electricity generation in 2023."
)

audit_relationship(
    19,
    "YES",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "PARTIAL",
    "YES",
    "YES",
    "NO",
    description_problem="The source states that digital market communication enables PV system operators to participate in markets, not that it supports the PHOTOVOLTAIK technology itself.",
    pipeline_stage="RELATIONSHIP_ENDPOINT_SELECTION",
    notes="The better target is PV-ANLAGENBETREIBER:INNEN or another market-participant stakeholder entity."
)

In [12]:
reviewed_relationships = relationship_review[
    relationship_review["primary_classification"] != ""
]

print("Reviewed relationships:", len(reviewed_relationships))
print()
print(reviewed_relationships["primary_classification"].value_counts())

Reviewed relationships: 20

primary_classification
DIRECTLY_SUPPORTED_BY_SOURCE         11
REASONABLE_INFERENCE_NOT_EXPLICIT     4
WRONG_DIRECTION_OR_ENDPOINTS          3
OVERGENERALIZED                       2
Name: count, dtype: int64


In [13]:
audit_relationship(
    20,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    description_problem="The description should avoid claiming specific innovation or deployment outcomes beyond the stated purpose of the funding programme.",
    pipeline_stage="RELATIONSHIP_DESCRIPTION_SUMMARIZATION",
    notes="The Climate and Energy Fund programme explicitly supports photovoltaic demonstration and lighthouse projects."
)

audit_relationship(
    21,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "NO",
    "YES",
    "YES",
    description_problem="The source presents expansion or application of eco-design requirements as a proposed or planned measure, not simply as an already established fact.",
    pipeline_stage="MODALITY_CLASSIFICATION",
    notes="REGULATES is appropriate, but PROPOSAL or PLANNED_ACTION would be more accurate than EXPLICIT_FACT."
)

audit_relationship(
    22,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly identifies complex approval procedures as a barrier to rapid photovoltaic expansion."
)

audit_relationship(
    23,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    description_problem="ASSOCIATED_WITH is valid but less informative than SUPPORTS or PROVIDES_EVIDENCE_FOR.",
    pipeline_stage="NONE",
    notes="E-Control flexibility studies are explicitly referenced as a basis for further flexibility-related implementation."
)

audit_relationship(
    24,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "NO",
    "YES",
    "YES",
    description_problem="Quality-assurance measures are presented as recommended or planned actions rather than completed factual effects.",
    pipeline_stage="MODALITY_CLASSIFICATION",
    notes="The relationship is supported, but RECOMMENDATION or PLANNED_ACTION is more accurate than EXPLICIT_FACT."
)

audit_relationship(
    25,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly reports more than 1 GW of new photovoltaic capacity in Austria in 2022."
)

audit_relationship(
    26,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly reports approximately 2.5 GW of photovoltaic capacity additions in Austria in 2023."
)

audit_relationship(
    27,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly reports approximately 6.3 TWh of photovoltaic electricity generation in Austria in 2023."
)

audit_relationship(
    28,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "NO",
    "YES",
    "YES",
    description_problem="The 30% value concerns a future 2040 target or projected contribution, not a present factual measurement.",
    pipeline_stage="MODALITY_CLASSIFICATION",
    notes="The relationship is source-supported, but TARGET or SCENARIO is more accurate than EXPLICIT_FACT."
)

audit_relationship(
    29,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly states that PV supplied nearly 10% of Austria's final electricity consumption at the end of 2023."
)

In [14]:
reviewed_relationships = relationship_review[
    relationship_review["primary_classification"] != ""
]

print("Reviewed relationships:", len(reviewed_relationships))
print()
print(reviewed_relationships["primary_classification"].value_counts())

Reviewed relationships: 30

primary_classification
DIRECTLY_SUPPORTED_BY_SOURCE         21
REASONABLE_INFERENCE_NOT_EXPLICIT     4
WRONG_DIRECTION_OR_ENDPOINTS          3
OVERGENERALIZED                       2
Name: count, dtype: int64


In [15]:
audit_relationship(
    30,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly states that PV covered approximately 2% of Austria's total energy demand in 2023."
)

audit_relationship(
    31,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly projects that photovoltaic installations will provide around 20% of Austria's total energy supply in 2040."
)

audit_relationship(
    32,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "NO",
    "NO",
    "YES",
    "YES",
    description_problem="The average annual expansion rate is a future requirement or target, not a present explicit fact.",
    pipeline_stage="RELATIONSHIP_TYPE_AND_MODALITY",
    notes="HAS_TARGET with TARGET or SCENARIO modality would be more precise than CONTRIBUTES_TO with EXPLICIT_FACT."
)

audit_relationship(
    33,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "NO",
    "YES",
    "YES",
    "YES",
    description_problem="Broad visibility is a qualitative characteristic of PV installations, not a measurement used to quantify photovoltaics.",
    pipeline_stage="RELATIONSHIP_TYPE_SELECTION",
    notes="HAS_CHARACTERISTIC or ASSOCIATED_WITH would be more suitable than MEASURED_BY."
)

audit_relationship(
    34,
    "PARTIAL",
    "OVERGENERALIZED",
    "YES",
    "PARTIAL",
    "YES",
    "YES",
    description_problem="The source connects the approximately 10% value to the end of 2023, while the generated relationship calls it current without retaining the year.",
    pipeline_stage="RELATIONSHIP_DESCRIPTION_SUMMARIZATION",
    notes="The numerical value is supported, but its temporal context was weakened."
)

audit_relationship(
    35,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    description_problem="This relationship substantially duplicates another 20% by 2040 PV-energy-supply relationship.",
    pipeline_stage="ENTITY_AND_RELATIONSHIP_RESOLUTION",
    notes="The claim is supported, but duplicated target entities create redundant relationships."
)

audit_relationship(
    36,
    "YES",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "YES",
    "YES",
    "NO",
    "YES",
    description_problem="The description says PV expansion contributes to sustainable and affordable electricity, but the stored edge points in the opposite direction.",
    pipeline_stage="RELATIONSHIP_DIRECTION",
    notes="The correct direction is PHOTOVOLTAIK CONTRIBUTES_TO NACHHALTIGE UND GÜNSTIGE STROMVERSORGUNG."
)

audit_relationship(
    37,
    "YES",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "YES",
    "YES",
    "NO",
    "YES",
    description_problem="The source says renewable-energy and PV expansion strengthen energy independence, but the stored edge points from energy independence to PV.",
    pipeline_stage="RELATIONSHIP_DIRECTION",
    notes="The direction should run from PHOTOVOLTAIK to ENERGIEUNABHÄNGIGKEIT ÖSTERREICHS."
)

audit_relationship(
    38,
    "YES",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "YES",
    "YES",
    "NO",
    "YES",
    description_problem="Jobs and domestic value creation are described as outcomes of PV expansion, but the stored edge points from jobs to PV.",
    pipeline_stage="RELATIONSHIP_DIRECTION",
    notes="The direction should run from PHOTOVOLTAIK to ARBEITSPLÄTZE DURCH PHOTOVOLTAIK."
)

audit_relationship(
    39,
    "NO",
    "UNSUPPORTED",
    "NO",
    "NO",
    "YES",
    "YES",
    description_problem="The strategy is not an alias of the speed and methodology of PV expansion. These are entirely different concepts.",
    pipeline_stage="RELATIONSHIP_TYPE_SELECTION",
    notes="The source discusses the importance of expansion speed and methodology, but it does not support an ALIAS_OF relationship."
)

In [16]:
reviewed_relationships = relationship_review[
    relationship_review["primary_classification"] != ""
]

print("Reviewed relationships:", len(reviewed_relationships))
print()
print(reviewed_relationships["primary_classification"].value_counts())

Reviewed relationships: 40

primary_classification
DIRECTLY_SUPPORTED_BY_SOURCE         26
WRONG_DIRECTION_OR_ENDPOINTS          6
REASONABLE_INFERENCE_NOT_EXPLICIT     4
OVERGENERALIZED                       3
UNSUPPORTED                           1
Name: count, dtype: int64


In [17]:
audit_relationship(
    40,
    "NO",
    "UNSUPPORTED",
    "NO",
    "NO",
    "YES",
    "YES",
    description_problem="Leading PV research institutions are not aliases of the broader category of all PV research institutions.",
    pipeline_stage="RELATIONSHIP_TYPE_SELECTION",
    notes="A subset or PART_OF relationship might be defensible, but ALIAS_OF is not supported."
)

audit_relationship(
    41,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="RED III explicitly amends Directive (EU) 2018/2001."
)

audit_relationship(
    42,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "NO",
    "YES",
    "YES",
    description_problem="Mobilising sealed or previously used surfaces is presented as a policy objective or proposed measure, not merely as an existing fact.",
    pipeline_stage="MODALITY_CLASSIFICATION",
    notes="APPLIES_TO is acceptable, but PLANNED_ACTION or RECOMMENDATION is more accurate than EXPLICIT_FACT."
)

audit_relationship(
    43,
    "PARTIAL",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "NO",
    "YES",
    "NO",
    "PARTIAL",
    description_problem="The source says clear municipal and state targets are required. It does not state that abstract targets apply to the entity GEMEINDE in this direction.",
    pipeline_stage="RELATIONSHIP_DIRECTION_AND_ENDPOINT_SELECTION",
    notes="A better relationship is GEMEINDE HAS_TARGET PV-AUSBAUZIEL."
)

audit_relationship(
    44,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly connects the Just Transition training initiative with training offensives in qualification alliances."
)

audit_relationship(
    45,
    "PARTIAL",
    "REASONABLE_INFERENCE_NOT_EXPLICIT",
    "YES",
    "NO",
    "YES",
    "YES",
    description_problem="E-Control is referenced through grid-connection actions and flexibility studies, but its description as a significant strategy actor is a synthesis.",
    pipeline_stage="RELATIONSHIP_INFERENCE",
    notes="The association is reasonable, but EXPLICIT_FACT overstates the directness of the claim."
)

audit_relationship(
    46,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly states that Agri-PV provides agriculture with new income opportunities and improves land-use efficiency."
)

audit_relationship(
    47,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "PARTIAL",
    "YES",
    "YES",
    description_problem="The output combines EXPLICIT_FACT with STATUS=PLANNED, creating ambiguity about whether the arrangement is existing or prospective.",
    pipeline_stage="MODALITY_AND_STATUS_CLASSIFICATION",
    notes="The active-customer use relationship is supported, but its legal or implementation status requires more precise representation."
)

audit_relationship(
    48,
    "PARTIAL",
    "REASONABLE_INFERENCE_NOT_EXPLICIT",
    "PARTIAL",
    "NO",
    "YES",
    "YES",
    description_problem="Environmental assessment requirements are relevant to permitting, but describing UVP categorically as constraining photovoltaics is an interpretation.",
    pipeline_stage="RELATIONSHIP_INFERENCE",
    notes="A more neutral relationship such as APPLIES_TO or REGULATES would be preferable."
)

audit_relationship(
    49,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly describes difficulties absorbing photovoltaic generation peaks in the public electricity grid."
)

In [18]:
reviewed_relationships = relationship_review[
    relationship_review["primary_classification"] != ""
]

print("Reviewed relationships:", len(reviewed_relationships))
print()
print(reviewed_relationships["primary_classification"].value_counts())

Reviewed relationships: 50

primary_classification
DIRECTLY_SUPPORTED_BY_SOURCE         32
WRONG_DIRECTION_OR_ENDPOINTS          7
REASONABLE_INFERENCE_NOT_EXPLICIT     6
OVERGENERALIZED                       3
UNSUPPORTED                           2
Name: count, dtype: int64


In [19]:
audit_relationship(
    50,
    "YES",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "NO",
    "YES",
    "NO",
    "YES",
    description_problem="The source describes Austrian companies as exporting PV technologies and components. The stored edge incorrectly runs from the export activity to the companies.",
    pipeline_stage="RELATIONSHIP_DIRECTION",
    notes="A better relationship is ÖSTERREICHISCHE UNTERNEHMEN EXPORTS PV-TECHNOLOGIE UND KOMPONENTEN."
)

audit_relationship(
    51,
    "YES",
    "REASONABLE_INFERENCE_NOT_EXPLICIT",
    "PARTIAL",
    "NO",
    "YES",
    "YES",
    description_problem="The strategy identifies measures intended to support climate neutrality, but the strategy document itself does not directly cause climate neutrality.",
    pipeline_stage="RELATIONSHIP_INFERENCE",
    notes="SUPPORTS or SETS_TARGET_FOR would be more cautious than CONTRIBUTES_TO, and the 2040 outcome is a target."
)

audit_relationship(
    52,
    "PARTIAL",
    "OVERGENERALIZED",
    "PARTIAL",
    "YES",
    "YES",
    "YES",
    description_problem="The source attributes approximately EUR 600 million jointly to the EAG and the Climate and Energy Fund; it does not allocate the entire amount to the Fund alone.",
    pipeline_stage="RELATIONSHIP_DESCRIPTION_GENERALIZATION",
    notes="The funding connection is valid, but the exact amount cannot be independently assigned to this single endpoint."
)

audit_relationship(
    53,
    "PARTIAL",
    "OVERGENERALIZED",
    "PARTIAL",
    "YES",
    "YES",
    "YES",
    description_problem="The approximately EUR 600 million is jointly associated with the EAG and Climate and Energy Fund, not exclusively disbursed under the EAG.",
    pipeline_stage="RELATIONSHIP_DESCRIPTION_GENERALIZATION",
    notes="GraphRAG split a combined funding statement into separate edges that each appear to claim the complete amount."
)

audit_relationship(
    54,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source explicitly states Austria's objective of nationally balanced 100% renewable electricity consumption by 2030."
)

audit_relationship(
    55,
    "YES",
    "OVERGENERALIZED",
    "NO",
    "YES",
    "YES",
    "YES",
    description_problem="The 41 TWh figure is identified as PV expansion potential in the NIP, not necessarily as an adopted Austrian target.",
    pipeline_stage="RELATIONSHIP_TYPE_SELECTION",
    notes="IDENTIFIES_POTENTIAL or HAS_POTENTIAL would be more accurate than HAS_TARGET."
)

audit_relationship(
    56,
    "YES",
    "WRONG_DIRECTION_OR_ENDPOINTS",
    "NO",
    "YES",
    "NO",
    "YES",
    description_problem="The action plan did not implement E-Control. E-Control published or developed the action plan.",
    pipeline_stage="RELATIONSHIP_DIRECTION",
    notes="The correct direction is E-CONTROL IMPLEMENTED_OR_PUBLISHED AKTIONSPLAN NETZANSCHLUSS."
)

audit_relationship(
    57,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source supports the connection between the EAG and financial deductions affecting ground-mounted PV installations."
)

audit_relationship(
    58,
    "YES",
    "DIRECTLY_SUPPORTED_BY_SOURCE",
    "YES",
    "YES",
    "YES",
    "YES",
    notes="The source supports EAG premiums or additional support for innovative photovoltaic system forms."
)

audit_relationship(
    59,
    "YES",
    "OVERGENERALIZED",
    "NO",
    "PARTIAL",
    "YES",
    "YES",
    description_problem="Being an official signatory to the Solar Charter does not necessarily mean that Austria implements the Charter as a programme.",
    pipeline_stage="RELATIONSHIP_TYPE_AND_STATUS",
    notes="SIGNED or PARTICIPATES_IN would be more accurate than IMPLEMENTS."
)

In [20]:
reviewed_relationships = relationship_review[
    relationship_review["primary_classification"] != ""
].copy()

print("Reviewed relationships:", len(reviewed_relationships))

print("\nPrimary classifications:")
print(
    reviewed_relationships[
        "primary_classification"
    ].value_counts()
)

print("\nPrimary classification percentages:")
print(
    reviewed_relationships[
        "primary_classification"
    ]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nSource support:")
print(
    reviewed_relationships[
        "source_supported"
    ].value_counts()
)

print("\nRelationship-type correctness:")
print(
    reviewed_relationships[
        "relation_type_correct"
    ].value_counts()
)

print("\nModality correctness:")
print(
    reviewed_relationships[
        "modality_correct"
    ].value_counts()
)

print("\nDirection correctness:")
print(
    reviewed_relationships[
        "direction_correct"
    ].value_counts()
)

print("\nEndpoint correctness:")
print(
    reviewed_relationships[
        "endpoints_correct"
    ].value_counts()
)

print("\nLikely pipeline stages:")
print(
    reviewed_relationships[
        "likely_pipeline_stage"
    ].value_counts()
)

Reviewed relationships: 60

Primary classifications:
primary_classification
DIRECTLY_SUPPORTED_BY_SOURCE         35
WRONG_DIRECTION_OR_ENDPOINTS          9
REASONABLE_INFERENCE_NOT_EXPLICIT     7
OVERGENERALIZED                       7
UNSUPPORTED                           2
Name: count, dtype: int64

Primary classification percentages:
primary_classification
DIRECTLY_SUPPORTED_BY_SOURCE         58.3
WRONG_DIRECTION_OR_ENDPOINTS         15.0
REASONABLE_INFERENCE_NOT_EXPLICIT    11.7
OVERGENERALIZED                      11.7
UNSUPPORTED                           3.3
Name: proportion, dtype: float64

Source support:
source_supported
YES        49
PARTIAL     9
NO          2
Name: count, dtype: int64

Relationship-type correctness:
relation_type_correct
YES        37
NO         15
PARTIAL     8
Name: count, dtype: int64

Modality correctness:
modality_correct
YES        37
NO         19
PARTIAL     4
Name: count, dtype: int64

Direction correctness:
direction_correct
YES        52
NO     

In [21]:
completed_path = (
    project_root
    / "pilot_02"
    / "inspection"
    / "05_relationship_review_completed.csv"
)

relationship_review.to_csv(
    completed_path,
    index=False,
    encoding="utf-8-sig",
)

print("Completed relationship audit saved to:")
print(completed_path)

Completed relationship audit saved to:
C:\Users\HP-ZBOOK i7\graphrag_test\pilot_02\inspection\05_relationship_review_completed.csv


# Pilot 2 — Relationship Extraction Audit

## Audit method

A systematic sample of 60 relationships was selected from the 643 relationships generated in Pilot 2.

The sample included:

- 25 highly connected relationships;
- 20 relationships involving TARGET or MARKET_METRIC entities;
- stratified examples from the controlled relationship vocabulary;
- additional randomly selected relationships where required.

Each relationship was traced to its linked text unit and assessed for:

- source support;
- explicitness versus inference;
- relationship-type correctness;
- modality correctness;
- direction correctness;
- endpoint correctness;
- description generalization;
- likely pipeline error stage.

GraphRAG's relationship `weight` was not interpreted as confidence or truth probability.

## Overall results

| Classification | Count | Percentage |
|---|---:|---:|
| Directly supported by source | 35 | 58.3% |
| Wrong direction or endpoints | 9 | 15.0% |
| Reasonable inference, not explicit | 7 | 11.7% |
| Overgeneralized | 7 | 11.7% |
| Unsupported | 2 | 3.3% |
| **Total** | **60** | **100.0%** |

## Source support

| Source support | Count | Percentage |
|---|---:|---:|
| Yes | 49 | 81.7% |
| Partial | 9 | 15.0% |
| No | 2 | 3.3% |

Most relationships were connected to genuine source concepts. However, source-supported endpoints did not guarantee that the stored relationship itself was correct.

## Controlled relationship-type quality

| Relationship-type correctness | Count | Percentage |
|---|---:|---:|
| Yes | 37 | 61.7% |
| Partial | 8 | 13.3% |
| No | 15 | 25.0% |

The revised prompt successfully generated useful controlled types such as:

- SUPPORTS;
- CONTRIBUTES_TO;
- MEASURED_BY;
- CONSTRAINS;
- HAS_TARGET;
- FUNDS;
- REGULATES;
- AMENDS.

Nevertheless, one quarter of the reviewed type labels were wrong.

Examples included:

- ALIAS_OF used between concepts that were not aliases;
- SUPPORTS used when REQUIRES was more accurate;
- MEASURED_BY used for a qualitative characteristic;
- HAS_TARGET used for a technical potential rather than an adopted target;
- IMPLEMENTS used where SIGNED or PARTICIPATES_IN was more accurate.

## Modality quality

| Modality correctness | Count | Percentage |
|---|---:|---:|
| Yes | 37 | 61.7% |
| No | 19 | 31.7% |
| Partial | 4 | 6.7% |

Modality remains a substantial bottleneck.

GraphRAG frequently labelled statements as `EXPLICIT_FACT` even when they represented:

- future targets;
- scenarios;
- proposals;
- planned actions;
- recommendations;
- reasonable interpretations.

Therefore, `EXPLICIT_FACT` cannot be treated as verified provenance or factual certainty.

## Direction and endpoint quality

### Direction

| Direction correctness | Count | Percentage |
|---|---:|---:|
| Yes | 52 | 86.7% |
| No | 7 | 11.7% |
| Partial | 1 | 1.7% |

### Endpoints

| Endpoint correctness | Count | Percentage |
|---|---:|---:|
| Yes | 57 | 95.0% |
| No | 2 | 3.3% |
| Partial | 1 | 1.7% |

Endpoint identification was generally strong. However, direction errors materially changed the meaning of several claims.

Examples included:

- sustainable electricity supply → photovoltaics, although the description stated that photovoltaics contributes to sustainable electricity supply;
- energy independence → photovoltaics, although PV expansion strengthens energy independence;
- jobs → photovoltaics, although jobs were described as an outcome of PV expansion;
- Action Plan Grid Connection → E-Control, although E-Control published or developed the action plan;
- export activity → Austrian companies, although Austrian companies perform the exporting.

This demonstrates that a plausible generated description can coexist with an incorrectly directed stored edge.

## Numerical and target relationships

Pilot 2 substantially improved the structural representation of numerical information.

Correctly captured examples included:

- 1 GW of annual PV additions in 2022;
- approximately 2.5 GW of annual PV additions in 2023;
- approximately 6.3 TWh of PV generation in 2023;
- approximately 10% of final electricity consumption in 2023;
- approximately 2% of total energy demand in 2023;
- approximately 20% of total energy supply projected for 2040;
- 100% nationally balanced renewable electricity consumption by 2030;
- the 21 TWh scenario requirement for 2030.

This directly addresses Pilot 1's most important recall failure, where all ten reviewed market and numerical relationships were missing.

Remaining numerical problems included:

- describing 11 TWh of energy generation as capacity;
- removing the 2023 reference from a 10% measurement and calling it current;
- treating the 41 TWh PV potential as an adopted target;
- assigning the complete EUR 600 million amount separately to both the EAG and Climate and Energy Fund, even though the source presented it jointly.

## Comparison with Pilot 1

The Pilot 1 relationship audit reviewed 42 relationships:

| Pilot 1 classification | Count | Percentage |
|---|---:|---:|
| Directly supported | 17 | 40.5% |
| Reasonable inference | 12 | 28.6% |
| Overgeneralized | 9 | 21.4% |
| Unsupported | 3 | 7.1% |
| Wrong direction or endpoints | 1 | 2.4% |

Pilot 2 reviewed 60 relationships:

| Pilot 2 classification | Count | Percentage |
|---|---:|---:|
| Directly supported | 35 | 58.3% |
| Reasonable inference | 7 | 11.7% |
| Overgeneralized | 7 | 11.7% |
| Unsupported | 2 | 3.3% |
| Wrong direction or endpoints | 9 | 15.0% |

Direct source support increased from approximately 40.5% to 58.3%. Inferences, generalizations and unsupported relationships all decreased.

The increase in identified direction errors should not automatically be interpreted as a general deterioration. Pilot 2 introduced more specific directed predicates, making direction errors easier to detect than in Pilot 1's broad natural-language relationships. The Pilot 2 sample also deliberately included many high-degree and numerical relationships.

The comparison is therefore indicative rather than a formal population estimate.

## Main remaining bottlenecks

The most frequent identified pipeline problems were:

1. relationship inference;
2. relationship-type selection;
3. relationship direction;
4. modality classification;
5. relationship-description summarization;
6. endpoint selection;
7. combined statements split into misleading separate edges;
8. entity and relationship resolution.

## Conclusion

Pilot 2 substantially improved relationship extraction, particularly for market metrics, targets and controlled semantic predicates.

However, structured labels do not make the relationships automatically trustworthy. Relationship type, modality, direction and numerical interpretation must still be verified against source evidence.

The recommended controlled-KG workflow remains:

GraphRAG relationship extraction  
→ source-evidence verification  
→ modality validation  
→ direction correction  
→ endpoint canonicalization  
→ controlled relationship import

Pilot 2 is a considerably stronger extraction layer than Pilot 1, but it is not yet a source-of-truth knowledge graph.